# Lab 9 — Shadow AI Log Audit  (v2 — realistic captured traffic)
### AI for Cybersecurity Professionals · Day 5 · Student Lab (60 min)

---

## What is this lab about?

**"Shadow AI"** is when employees use AI tools (ChatGPT, Claude, Gemini, Copilot, Midjourney, and so on) that **nobody in IT or security approved**. It is the AI version of "Shadow IT."

Why should a security team care? Because when an employee pastes a document into an outside AI website, that data **leaves the company** — it crosses the internet to a third party. If that document contains customer records, source code, or financial data, you may have just had a data leak that **no firewall alert ever fired on**, because the traffic looks like ordinary web browsing.

Your job is the job of a real security analyst on a **Monday-morning review**: you are handed **captured network traffic** and asked one question:

> *"Is anyone using AI tools they shouldn't be — and is any sensitive data walking out the door?"*

## What's new in v2: the logs look like real captured traffic

Version 1 used tidy domain names. Real firewall / proxy / NetFlow captures don't — they record the **5-tuple** of each connection, using **IP addresses and port numbers**, not names. So v2 gives you rows that look like what you'd actually pull off the wire:

| Column | Example | Meaning |
|---|---|---|
| `timestamp` | 2026-06-01 17:03 | when the packet was seen |
| `src_ip` | 10.20.30.108 | the **internal** computer that started the connection |
| `src_port` | 57526 | the caller's **ephemeral** port — random, throwaway, **we ignore it** |
| `dst_ip` | 203.0.113.18 | the **server** being contacted (this is the important one) |
| `dst_port` | 443 | the **service** port (443 = HTTPS, 53 = DNS, 465/993/995 = email) |
| `protocol` | TCP / UDP | transport protocol |
| `bytes_transferred` | 31955083 | how much data moved (our exfiltration signal) |
| `info` | Standard query A claude.ai | extra detail (blank for web) |

Because the capture shows **IPs, not names**, we keep an **IP lookup key** — a table that maps each made-up service IP back to its real service and tells us whether that service is *approved* or a *prohibited AI tool*. That lookup is exactly the kind of **threat-intelligence feed** a security team maintains. Our detection matches on **destination IP**, then uses the lookup to turn IPs back into readable names for the report.

### Two facts about ports (so the logs make sense)
- **Well-known / destination ports** identify the *service*: `443` HTTPS (web), `53` DNS, `465` SMTPS (send email), `993` IMAPS and `995` POP3S (retrieve email).
- **Source ports** are **ephemeral** — the operating system grabs a random high number (49152–65535) for each outbound connection. They carry no meaning for detection, so **our tool ignores `src_port` entirely.**

The capture also contains realistic **background noise** — about 20 email packets and 20 DNS lookups to Google's `8.8.8.8` — which a good analyst learns to recognise and set aside. Our tool treats email and DNS as noise and does **not** flag them.

## Environment
- **Google Colab**, free tier, **CPU runtime** — no GPU and **no API key** needed.

## How to use this notebook
Click **▶ (Run)** on each cell top to bottom, in order. The comments in each code cell explain every line in plain English.


---
## Section 0 — Set up our tools (import the libraries)

A **library** is a bundle of pre-written code we bring in with `import`. Run the cell below; it loads the four libraries this lab needs and explains what each is for.


In [ ]:
# ============================================================
# SECTION 0 — SET UP OUR TOOLS (import the libraries)
# ============================================================

import numpy as np      # numpy = "Numerical Python". Fast math on big lists of numbers.
                        # We use it to create random — but REPEATABLE — fake traffic
                        # (random users, IPs, ports, and byte counts).

import pandas as pd     # pandas = the "spreadsheet of Python". Its DataFrame is a table
                        # with rows and named columns, like Excel. Our whole audit runs on one.

import matplotlib.pyplot as plt   # matplotlib = the charting library (bars, pies). "plt" for short.

import seaborn as sns   # seaborn makes matplotlib charts look clean with no extra effort.

sns.set_theme(style="whitegrid")   # give every chart a consistent, tidy style

# ----- A note about versions (reproducibility) -----
# Colab already has all four libraries, so we skip "pip install" (and avoid a restart).
# The original lab spec pinned: pandas==2.2.1, numpy==1.26.4, matplotlib, seaborn.
# For byte-for-byte reproducibility elsewhere you COULD run the next line, but it forces a
# Colab runtime restart, so we leave it commented:
# !pip install pandas==2.2.1 numpy==1.26.4 matplotlib seaborn

print("All libraries loaded. Ready to go!")

---
## Section 1 — Build the IP address "lookup key" (our intelligence feed)

Because captured traffic shows **IP addresses instead of names**, we first build the reference table that lets us make sense of those IPs. For each service we invent an IP address and record whether it is an **approved** business site or a **prohibited AI tool**.

> **Why the odd IP ranges?** We use `203.0.113.x` and `198.51.100.x`. These are official *documentation* ranges (reserved by the standards body TEST-NET-3 / TEST-NET-2) — they look like real public IPs but can never belong to a real company, so they are perfectly safe to use in training material. Google's real public DNS, `8.8.8.8`, is the one genuine address we keep, because everyone recognises it.

Run the cell to build the lookup and print it.


In [ ]:
# ============================================================
# SECTION 1 — THE IP LOOKUP KEY (services -> IP addresses)
# ============================================================

# Our 10 employees, and the INTERNAL IP address of each person's computer.
users = [f"user_{i:02d}" for i in range(1, 11)]
user_ip = {u: f"10.20.30.{100 + i}" for i, u in enumerate(users, start=1)}   # 10.20.30.101 ... .110

# Which department each employee is on (2 per department).
user_dept = {
    "user_01": "Engineering", "user_02": "Engineering",
    "user_03": "Sales",       "user_04": "Sales",
    "user_05": "Marketing",   "user_06": "Marketing",
    "user_07": "Finance",     "user_08": "Finance",
    "user_09": "HR",          "user_10": "HR",
}

# The PROHIBITED AI tools we are hunting for (the "shadow AI").
ai_domains = [
    "chat.openai.com", "claude.ai", "gemini.google.com", "copilot.microsoft.com",
    "midjourney.com", "character.ai", "huggingface.co", "perplexity.ai", "poe.com",
]

# APPROVED, everyday business sites (the normal "background noise").
normal_domains = [
    "github.com", "stackoverflow.com", "google.com", "office.com", "salesforce.com",
    "aws.amazon.com", "slack.com", "zoom.us", "wikipedia.org", "atlassian.net",
    "linkedin.com", "dropbox.com", "docs.google.com", "outlook.com",
]

# Give every service a made-up but realistic public IP address:
#   AI tools     -> 203.0.113.10, .11, .12 ...
#   approved     -> 198.51.100.10, .11, .12 ...
ai_ip     = {d: f"203.0.113.{10 + i}"  for i, d in enumerate(ai_domains)}
normal_ip = {d: f"198.51.100.{10 + i}" for i, d in enumerate(normal_domains)}

# Two handy lookup directions:
service_to_ip = {**ai_ip, **normal_ip}                 # name -> IP  (used to BUILD traffic)
ip_to_service = {ip: name for name, ip in service_to_ip.items()}   # IP -> name (used to READ traffic)

# The set of IPs that belong to AI tools -- this is what our detector matches against.
ai_ips = set(ai_ip.values())

# Infrastructure IPs for the background-noise traffic:
MAIL_IP = "198.51.100.25"   # the company mail server (send/receive email)
DNS_IP  = "8.8.8.8"         # Google Public DNS (real, well-known address)

MB = 1024 * 1024            # 1 megabyte in bytes = 1,048,576 -- used all over this lab

# Show the lookup key as a neat table, sorted by category:
lookup = pd.DataFrame(
    [{"service": d, "ip_address": ip, "category": "PROHIBITED (AI tool)"} for d, ip in ai_ip.items()] +
    [{"service": d, "ip_address": ip, "category": "approved"}             for d, ip in normal_ip.items()]
)
print("IP LOOKUP KEY (our intelligence feed):")
lookup

---
## Section 2 — Generate the captured traffic (web + email + DNS)

Now we synthesise ~**540 packets** of realistic capture:

- **~500 web connections** (destination port **443**) — about 15% go to AI tools, and a few of those are **huge uploads** (our planted data-exfiltration cases).
- **~20 email packets** — sending on **465 (SMTPS)** and retrieving on **993 (IMAPS)** and **995 (POP3S)**, to the mail server.
- **~20 DNS lookups** — to **8.8.8.8 port 53**, each querying one of the domains in our program.

**The magic word: `seed`.** A fixed seed of `42` makes the "random" data identical every run, so your results match the answer key and your classmates. That is *reproducibility* — essential when a report might end up in an investigation.


In [ ]:
# ============================================================
# SECTION 2 — GENERATE THE CAPTURED TRAFFIC
# ============================================================

# One random-number generator with a FIXED seed -> same data every run.
rng = np.random.default_rng(42)

# ---------- (a) WEB TRAFFIC: ~500 connections on port 443 (HTTPS) ----------
NW = 500
web_users = rng.choice(users, size=NW)                    # who made each request
is_ai     = rng.random(NW) < 0.15                         # ~15% go to an AI tool
service   = np.where(is_ai,                               # pick the destination service name
                     rng.choice(ai_domains, size=NW),
                     rng.choice(normal_domains, size=NW))

# Byte counts: normal browsing is small; AI chats a bit bigger; a few AI rows are HUGE uploads.
normal_bytes  = rng.integers(1_000, 800_000, size=NW)      # 1 KB - 800 KB
ai_small      = rng.integers(50_000, 3_000_000, size=NW)   # 50 KB - 3 MB (typical AI use)
is_big_upload = is_ai & (rng.random(NW) < 0.12)            # ~12% of AI rows are large
ai_big        = rng.integers(11 * MB, 40 * MB, size=NW)    # 11 MB - 40 MB (suspicious!)
web_bytes = np.where(is_ai, np.where(is_big_upload, ai_big, ai_small), normal_bytes)

web_srcport = rng.integers(49152, 65535, size=NW)          # random EPHEMERAL source ports (ignored)
web_min     = rng.integers(0, 30 * 24 * 60, size=NW)       # a random minute within ~30 days

web = pd.DataFrame({
    "user_id": web_users,
    "src_port": web_srcport,
    "dst_ip": [service_to_ip[s] for s in service],         # look up each service's IP
    "dst_port": 443,                                        # 443 = HTTPS (all web traffic)
    "protocol": "TCP",
    "bytes_transferred": web_bytes,
    "info": "",
    "_min": web_min,
})

# ---------- (b) EMAIL TRAFFIC: ~20 packets (send 465, retrieve 993 / 995) ----------
NE = 20
email_port = rng.choice([465, 993, 995], size=NE)          # SMTPS / IMAPS / POP3S
port_label = {465: "SMTP send (SMTPS)", 993: "IMAP fetch (IMAPS)", 995: "POP3 fetch (POP3S)"}
email = pd.DataFrame({
    "user_id": rng.choice(users, size=NE),
    "src_port": rng.integers(49152, 65535, size=NE),
    "dst_ip": MAIL_IP,                                      # all email goes to the mail server
    "dst_port": email_port,
    "protocol": "TCP",
    "bytes_transferred": rng.integers(2_000, 200_000, size=NE),
    "info": [port_label[p] for p in email_port],
    "_min": rng.integers(0, 30 * 24 * 60, size=NE),
})

# ---------- (c) DNS TRAFFIC: ~20 lookups to 8.8.8.8 port 53 ----------
ND = 20
dns_query = rng.choice(ai_domains + normal_domains, size=ND)   # each looks up a domain we know
dns = pd.DataFrame({
    "user_id": rng.choice(users, size=ND),
    "src_port": rng.integers(49152, 65535, size=ND),
    "dst_ip": DNS_IP,                                      # 8.8.8.8 = Google Public DNS
    "dst_port": 53,                                        # 53 = DNS
    "protocol": "UDP",                                     # DNS lookups use UDP
    "bytes_transferred": rng.integers(60, 200, size=ND),  # DNS packets are tiny
    "info": [f"Standard query A {q}" for q in dns_query],
    "_min": rng.integers(0, 30 * 24 * 60, size=ND),
})

# ---------- Combine all three into one capture ----------
df = pd.concat([web, email, dns], ignore_index=True)
df["src_ip"] = df["user_id"].map(user_ip)                 # fill in each user's internal IP
df["department"] = df["user_id"].map(user_dept)
start = np.datetime64("2026-06-01T09:00:00")
df["timestamp"] = start + df["_min"].to_numpy().astype("timedelta64[m]")
df = df.sort_values("timestamp").reset_index(drop=True)

# Put the columns in a natural "capture" order and drop the helper "_min".
df = df[["timestamp", "src_ip", "src_port", "dst_ip", "dst_port",
         "protocol", "bytes_transferred", "info", "user_id", "department"]]

print(f"Captured {len(df)} packets "
      f"({(df['dst_port']==443).sum()} web, "
      f"{df['dst_ip'].eq(MAIL_IP).sum()} email, "
      f"{(df['dst_port']==53).sum()} DNS). First 8 rows:")
df.head(8)

---
## Section 3 — Detect Shadow AI by destination IP

Here is the core detection. In real captured traffic there are no names — so we match each packet's **`dst_ip`** against our set of known **AI-tool IPs** (and require **port 443**, i.e. an actual web connection, not a DNS lookup *about* an AI domain).

Notice we **never look at `src_port`** — it is random noise. And email/DNS packets go to the mail server or `8.8.8.8`, so they are automatically *not* flagged. Once flagged, we use the lookup key to turn the IP back into a readable **service** name for the report.


In [ ]:
# ============================================================
# SECTION 3 — DETECT SHADOW AI (match on destination IP)
# ============================================================

# A packet is "shadow AI" if its destination IP is a known AI-tool IP
# AND it is a real web connection (port 443). We ignore src_port completely.
df["is_ai_tool"] = df["dst_ip"].isin(ai_ips) & (df["dst_port"] == 443)

# Build a table of just the AI connections, and resolve each IP back to its name.
ai = df[df["is_ai_tool"]].copy()
ai["service"] = ai["dst_ip"].map(ip_to_service)           # 203.0.113.18 -> "huggingface.co"

print(f"Out of {len(df)} captured packets, {len(ai)} were connections to prohibited AI tools.")
print("(Email and DNS packets were correctly ignored as background noise.)")
ai[["timestamp", "src_ip", "dst_ip", "service", "bytes_transferred"]].head()

---
## Section 4 — Aggregate AI usage per employee

"Aggregate" means **group and total up**. We group the AI connections by user and, for each person, count their AI sessions and sum the bytes they sent to AI tools. This `groupby` is the workhorse of all log analysis.


In [ ]:
# ============================================================
# SECTION 4 — AGGREGATE AI USAGE PER USER
# ============================================================

per_user = ai.groupby("user_id").agg(
    ai_sessions=("dst_ip", "size"),                 # how many AI connections
    bytes_to_ai=("bytes_transferred", "sum"),       # total bytes sent to AI tools
).sort_values("ai_sessions", ascending=False)

per_user["MB_to_ai"] = (per_user["bytes_to_ai"] / MB).round(1)   # friendly megabytes column

print("AI usage per employee (most active first):")
per_user

---
## Section 5 — The Top 5 shadow AI users (and their favourite tool)

Leadership wants the **short list**. We take the five heaviest AI users and, for each, find their **preferred tool** — the AI service they connected to most (the statistical *mode*).


In [ ]:
# ============================================================
# SECTION 5 — TOP 5 SHADOW AI USERS + PREFERRED TOOL
# ============================================================

top5 = per_user.head(5).copy()

# For each user, the most-used AI service (resolved from IP via the lookup key).
preferred = ai.groupby("user_id")["service"].agg(lambda s: s.mode().iat[0])
top5["preferred_tool"] = [preferred[u] for u in top5.index]

print("TOP 5 SHADOW AI USERS:")
top5[["ai_sessions", "MB_to_ai", "preferred_tool"]]

---
## Section 6 — Which departments are the biggest risk?

Managers think in **teams**. We roll AI usage up by department so we can say *"Marketing is our biggest Shadow-AI hotspot."* We visualise this as a pie chart in Section 8.


In [ ]:
# ============================================================
# SECTION 6 — DEPARTMENT RISK ANALYSIS
# ============================================================

dept_risk = ai.groupby("department").agg(
    ai_sessions=("dst_ip", "size"),
    bytes_to_ai=("bytes_transferred", "sum"),
).sort_values("ai_sessions", ascending=False)
dept_risk["MB_to_ai"] = (dept_risk["bytes_to_ai"] / MB).round(1)

print("Shadow AI usage by department (highest first):")
dept_risk

---
## Section 7 — Flag large uploads (over 10 MB) — possible data exfiltration

This is the finding a security team cares about most. A short AI *chat* moves a few kilobytes. But a **30 MB upload** to an outside AI site could be an entire customer database or code repository leaving the company. We flag every AI connection that moved **more than 10 MB** (`10 * 1024 * 1024` bytes). Each flagged row is a lead to investigate.


In [ ]:
# ============================================================
# SECTION 7 — FLAG LARGE UPLOADS (>10 MB) TO AI TOOLS
# ============================================================

THRESHOLD = 10 * MB   # 10 MB in bytes = 10,485,760

big_uploads = ai[ai["bytes_transferred"] > THRESHOLD].copy()
big_uploads["MB"] = (big_uploads["bytes_transferred"] / MB).round(1)

print(f"Found {len(big_uploads)} large upload(s) over 10 MB to AI tools — investigate these:")
big_uploads[["timestamp", "src_ip", "user_id", "department", "dst_ip", "service", "MB"]] \
    .sort_values("MB", ascending=False)

---
## Section 8 — The Shadow AI Usage Report (summary + charts)

Finally we pull everything into a short summary and three charts — the one-page deliverable you'd hand a manager:

- **Bar** — which AI tools are used most.
- **Bar** — how much data each employee sent to AI tools.
- **Pie** — each department's share of Shadow AI activity.


In [ ]:
# ============================================================
# SECTION 8 — SUMMARY REPORT + CHARTS
# ============================================================

total_sessions = len(ai)
unique_users   = ai["user_id"].nunique()
total_MB       = round(ai["bytes_transferred"].sum() / MB, 1)
top_tools      = ai["service"].value_counts().head(3)

print("=" * 48)
print("        SHADOW AI USAGE REPORT")
print("=" * 48)
print(f"Packets captured             : {len(df)}")
print(f"Connections to AI tools      : {total_sessions}")
print(f"Employees using AI tools     : {unique_users} of {len(users)}")
print(f"Total data sent to AI tools  : {total_MB} MB")
print(f"Large uploads (>10 MB)       : {len(big_uploads)}")
print("Top 3 AI tools by usage:")
for tool, count in top_tools.items():
    print(f"   - {tool}: {count} sessions")
print("=" * 48)

# ---- Chart 1: AI tool usage by frequency ----
plt.figure(figsize=(8, 4))
ai["service"].value_counts().plot(kind="bar", color="#1E5199")
plt.title("Shadow AI Tool Usage by Frequency")
plt.xlabel("AI tool"); plt.ylabel("number of sessions")
plt.tight_layout(); plt.show()

# ---- Chart 2: data sent to AI tools, per user ----
plt.figure(figsize=(8, 4))
(per_user["bytes_to_ai"] / MB).sort_values(ascending=False).plot(kind="bar", color="#00838F")
plt.title("Data Sent to AI Tools per User")
plt.xlabel("user"); plt.ylabel("megabytes (MB)")
plt.tight_layout(); plt.show()

# ---- Chart 3: department share of Shadow AI ----
plt.figure(figsize=(5, 5))
dept_risk["ai_sessions"].plot(kind="pie", autopct="%1.0f%%")
plt.title("Department Share of Shadow AI Sessions")
plt.ylabel("")
plt.tight_layout(); plt.show()

print("\nReport complete. You just ran a real Shadow-AI audit on captured traffic, end to end.")

---
## What you just learned

- **Real captures use IPs and ports, not names.** You detect Shadow AI by matching **destination IP** against an intelligence list, then resolve IPs back to names with a **lookup key**.
- **Source ports are noise.** Ephemeral high ports change every connection — ignore them. It's the **destination** IP + port that identify the service.
- **Not all traffic is web.** You separated real signal (port 443 to AI IPs) from background noise (email on 465/993/995, DNS on 8.8.8.8:53).
- **Bytes tell a story.** Small = chatting; large uploads = possible data leaving the company.
- **Reproducibility matters.** A fixed `seed` means anyone can re-run your work and get the same answer.

### Try it yourself (optional stretch goals)
1. Change the AI rate from `0.15` to `0.30` and re-run. How does the report change?
2. Add a new AI tool: add a domain to `ai_domains`, give it an IP, and watch it appear.
3. **Bonus detection:** the DNS packets include lookups of AI domains. Write a few lines to flag DNS queries (port 53) whose `info` field contains an AI domain — a second, independent way to catch Shadow AI.
4. Lower the large-upload threshold to **5 MB** and see how many more connections get flagged.


---
---
# 🔒 Instructor Answer Key
*(Reveal after the lab. Numbers assume the notebook is run top-to-bottom, unchanged, with `seed = 42`. NumPy's `default_rng` stream is stable, so results match on current Colab; a different NumPy **major** version could shift them slightly. The web-traffic figures are identical to Lab 9 v1 — only the log *format* changed.)*

**Headline numbers**
- Packets captured: **540** (500 web + 20 email + 20 DNS).
- Connections to AI tools: **86**.
- Employees using AI tools: **10 of 10**.
- Total data sent to AI tools: **≈ 362 MB**.
- Large uploads over 10 MB: **9** (the planted exfiltration cases).

**Top 5 shadow AI users (by sessions)**

| Rank | User | Internal IP | AI sessions | Preferred tool |
|---|---|---|---|---|
| 1 | user_05 | 10.20.30.105 | 15 | huggingface.co |
| 2 | user_08 | 10.20.30.108 | 13 | gemini.google.com |
| 3 | user_06 | 10.20.30.106 | 9 | midjourney.com |
| 4 | user_01 | 10.20.30.101 | 8 | character.ai |
| 5 | user_04 | 10.20.30.104 | 8 | character.ai |

**Department risk (by AI sessions):** Marketing **24** → Finance **21** → Engineering **14** ≈ Sales **14** → HR **13**. *Marketing is the top hotspot.*

**Top 3 tools by usage:** character.ai (**15**) → huggingface.co (**13**) → claude.ai (**11**).

**Teaching points to land**
- **Detection is on `dst_ip`, not names.** Drive home that the analyst's power comes from the **lookup key** (the intelligence feed) — bad IPs are worthless without it, and names are what leadership understands.
- **`src_port` is a distractor.** If a student tries to analyse source ports, that's the teachable moment: ephemeral ports are noise.
- **Noise vs signal.** Ask why the 20 DNS and 20 email packets were *not* flagged (different destination IPs/ports), and note the stretch goal: DNS-to-AI-domain is a legitimate *second* detection method.
- **The >10 MB flag is the money finding** — the 9 large uploads are what a SOC escalates.
- **Governance tie-in (Day 5):** Shadow AI is **both** a security risk **and** a regulatory one (EU AI Act / data residency).
